# Phase 3 — Calibration analysis

This notebook **displays** results; it does not compute its own. Every number
and every figure comes from `src/analysis/`, the same code `make analyze` and
`make figures` run, so a chart here can never disagree with the report.

If you want to change a result, change the module or `config.yaml` — not a cell.

- Method and its justification: `docs/methodology.md`, `docs/adr/005-bucketing-and-tests.md`
- Plain-English results: `docs/findings.md`


In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.analysis.calibration import (
    brier_decomposition,
    calibration_table,
    logistic_calibration,
)
from src.analysis.segmentation import bias_by_category, bias_by_lifetime
from src.config import get_config

config = get_config()
df = pd.read_parquet(config.clean.processed_path)
print(f"{len(df):,} contracts, {df.event_ticker.nunique():,} events")
df.head()

## 1. The headline

Brier score and its Murphy decomposition. `binned_brier` is the score the
three-term identity closes on exactly; `brier` is the real score on unbinned
prices, and `binning_residual` is the gap — see the docstring for why both.


In [ ]:
brier_decomposition(df)

## 2. Calibration table

Intervals and p-values cluster on `event_ticker`. `deff` above 1 is the factor
by which assuming independence would have narrowed the interval; `q_value` is
Benjamini-Hochberg-corrected across the ten buckets.


In [ ]:
table = calibration_table(df)
table

In [ ]:
display(Image('reports/figures/01_reliability_diagram.png'))
display(Image('reports/figures/02_bias_by_bucket.png'))

## 3. Logistic calibration

The bucket-free read, so the shape above cannot be an artefact of where the
bin edges fell. Perfect calibration is slope 1, intercept 0. **Slope above 1
is the favorite-longshot direction** — true probabilities more extreme than
prices; see the docstring for the derivation, it is easy to reverse.


In [ ]:
logistic_calibration(df)

## 4. Where the bias lives

Every segment × bucket test below sits in **one** correction family. A cell
with fewer than `min_events_per_bucket` events keeps its estimate but is never
tested (`underpowered`), because clustered inference is governed by the number
of events, not contracts — and Politics has 16 events in total.


In [ ]:
bias_by_category(df)

In [ ]:
display(Image('reports/figures/03_bias_by_category.png'))

### By market lifetime

Not time-to-resolution, which is ~1h for every row by construction (ADR 003
prices at T-1h). Lifetime is how long the market traded *before* that moment.


In [ ]:
bias_by_lifetime(df)

In [ ]:
display(Image('reports/figures/04_bias_by_lifetime.png'))